In [ ]:
from pathlib import Path
import os
import sys
import ast
import hashlib
import json
import importlib

REPO = Path("/workspace/ProtoSAM-repro")
os.chdir(REPO)

assert Path(sys.executable) == Path(
    "/workspace/miniforge3/envs/protosam/bin/python"
), f"Kernel incorreto: {sys.executable}"

MODULOS = REPO / "exp/eft_validacao_polipos_v1"

for pasta in (REPO, MODULOS):
    if str(pasta) not in sys.path:
        sys.path.insert(0, str(pasta))

PILOTO = REPO / "reproducao/eft_tiny_piloto_20261011T152746615659Z"
CHECKPOINT = (
    REPO / "reproducao/eft_tiny_2000_v2_20261011T204725195685Z"
    / "checkpoint_2000.pt"
)
REFERENCIA = (
    REPO / "reproducao"
    / "eft_tiny_referencia_driver_atual_20261012T215801165279Z"
)
DADOS = (
    REPO / "reproducao"
    / "eft_sp_piloto_dados_20261011T152315570066Z"
)

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha256(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

for p in (
    PILOTO / "config.json",
    CHECKPOINT,
    REFERENCIA / "config.json",
    DADOS / "divisao.json",
):
    assert p.is_file(), f"Arquivo ausente: {p}"

ref = ler_json(REFERENCIA / "config.json")
assert sha256(CHECKPOINT) == ref["checkpoint_eft_sha256"]

origem = MODULOS / "validar_polipos.py"
destino = MODULOS / "validar_multissuporte_v1.py"
codigo = origem.read_text(encoding="utf-8")

def trocar(antigo, novo):
    global codigo
    quantidade = codigo.count(antigo)
    assert quantidade == 1, (
        f"Versão do avaliador diferente da esperada "
        f"({quantidade} ocorrências): {antigo[:100]}"
    )
    codigo = codigo.replace(antigo, novo, 1)

trocar(
    "def executar(piloto, retomar=None):",
    """def executar(piloto, suporte_override, checkpoint_externo,
             referencia, destino_novo, retomar=None):"""
)

trocar(
    "suporte_meta = manifesto['suporte_k1']",
    "suporte_meta = dict(suporte_override)"
)

trocar(
    "assert Path(suporte_meta['imagem']).name == "
    "'cju3xtufwiv9c0818djsc4cqd.png'",
    """permitidos = {
        str(Path(r['imagem']).resolve())
        for r in manifesto['pool_adaptacao']
    } | {str(Path(manifesto['suporte_k1']['imagem']).resolve())}
    assert str(Path(suporte_meta['imagem']).resolve()) in permitidos"""
)

trocar(
    "assert int((suporte['label'] > .5).sum()) == 51940, "
    "'Suporte difere da referência.'",
    """area = int((suporte['label'] > .5).sum())
    assert 0 < area < 1024 * 1024, 'Suporte sem as duas classes.'
    if Path(suporte_meta['imagem']).name == 'cju3xtufwiv9c0818djsc4cqd.png':
        assert area == 51940, 'Suporte original difere da referência.'"""
)

trocar(
    "ckpt_eft = piloto / 'checkpoint_latest.pt'",
    "ckpt_eft = Path(checkpoint_externo).resolve()"
)

trocar(
    "assert salvo['passo'] == 200 and salvo['config'] == cfg_piloto",
    "assert salvo['passo'] == 2000, 'Checkpoint não é de 2.000 passos.'"
)

trocar(
    "'finalidade':'Validação anotada antes/depois do piloto; não é teste final'",
    "'finalidade':'Robustez a suportes: inicial versus EFT 2000; validação'"
)

linha_pasta = (
    "pasta = Path(retomar).resolve() if retomar else "
    "REPO/'reproducao'/f'eft_{variante}_validacao_polipos_{stamp()}'"
)

trocar(
    linha_pasta,
    """# Mantém o protocolo e os pesos da referência já avaliada.
    ref = ler(Path(referencia) / 'config.json')
    chaves = (
        'variante', 'episodios', 'resolucao_encoder', 'canvas',
        'sam', 'selecao_sam', 'alp_inferencia', 'grade_features',
        'kernel_configurado', 'checkpoint_eft_sha256',
        'checkpoint_original_sha256', 'checkpoint_sam_sha256',
        'torch', 'cuda', 'versoes', 'cudnn',
        'tf32_matmul', 'tf32_cudnn',
    )
    for chave in chaves:
        assert config[chave] == ref[chave], f'Referência incompatível: {chave}'

    for fonte, digest in ref['fontes_sha256'].items():
        if fonte in config['fontes_sha256']:
            assert config['fontes_sha256'][fonte] == digest, fonte

    config['passo'] = 2000
    config['referencia'] = str(Path(referencia).resolve())
    config['gpu_driver_referencia'] = ref['gpu_driver']
    print('GPU da referência:', ref['gpu_driver'], flush=True)
    print('GPU atual:', config['gpu_driver'], flush=True)

    pasta = Path(retomar or destino_novo).resolve()"""
)

ast.parse(codigo)

if destino.exists():
    assert destino.read_text(encoding="utf-8") == codigo, (
        "O módulo existente tem outro conteúdo. Não foi sobrescrito."
    )
else:
    destino.write_text(codigo, encoding="utf-8")

importlib.invalidate_caches()
import validar_multissuporte_v1 as avaliador
avaliador = importlib.reload(avaliador)

assert Path(avaliador.__file__).resolve() == destino.resolve()

print("Python:", sys.executable)
print("Avaliador:", destino)
print("Checkpoint:", CHECKPOINT)
print("Preparação concluída. Avaliação ainda não iniciada.")

In [ ]:
import random
import pandas as pd

PASTA_DADOS = REPO / "exp/eft_sp_piloto_v1"
assert (PASTA_DADOS / "eft_dados.py").is_file(), (
    f"Não encontrei: {PASTA_DADOS / 'eft_dados.py'}"
)
if str(PASTA_DADOS) not in sys.path:
    sys.path.insert(0, str(PASTA_DADOS))

EXECUCAO = REPO / "reproducao/multissuporte_tiny_eft2000_kvasir_v1"
EXECUCAO.mkdir(parents=True, exist_ok=True)
ARQUIVO_PLANO = EXECUCAO / "plano.json"

cfg_piloto = ler_json(PILOTO / "config.json")
assert Path(cfg_piloto["dados"]).resolve() == DADOS.resolve()
assert sha256(DADOS / "planos.json") == cfg_piloto["planos_sha256"]

planos = ler_json(DADOS / "planos.json")
divisao = ler_json(DADOS / "divisao.json")

# Verifica a divisão completa contra a assinatura da preparação.
from eft_dados import assinatura
assert assinatura(divisao) == planos["config"]["divisao_sha256"]

manifesto_path = Path(planos["config"]["manifesto"])
assert sha256(manifesto_path) == planos["config"]["manifesto_sha256"]
manifesto = ler_json(manifesto_path)

def caminho(p):
    return str(Path(p).resolve())

pool = {
    caminho(r["imagem"]): r
    for r in manifesto["pool_adaptacao"]
}

validacao = {
    caminho(r["imagem"])
    for r in divisao
    if r["split"] == "validacao"
}
treino = {
    caminho(r["imagem"])
    for r in divisao
    if r["split"] == "treino"
}
teste = {caminho(r["imagem"]) for r in manifesto["teste"]}

assert len(treino) == 1304
assert len(validacao) == 145
assert validacao == {caminho(r["imagem"]) for r in planos["validacao"]}
assert not treino & validacao
assert not (treino | validacao) & teste

# No conjunto auditado, os nomes cju.../cjy... identificam Kvasir.
# Nomes são usados aqui para selecionar a origem, não índices do loader.
candidatos = sorted(
    [
        pool[p] for p in treino
        if Path(p).stem.lower().startswith("cj")
    ],
    key=lambda r: caminho(r["imagem"]),
)

assert len(candidatos) >= 19
SEMENTE = 20261014
sorteados = random.Random(SEMENTE).sample(candidatos, 19)

selecionados = [manifesto["suporte_k1"]] + sorteados
assert len({caminho(r["imagem"]) for r in selecionados}) == 20

suportes = []
for i, r in enumerate(selecionados):
    p = caminho(r["imagem"])
    assert p not in validacao | teste
    if i > 0:
        assert p in treino

    suportes.append({
        "id": f"S{i:02d}",
        "tipo": "original_reservado" if i == 0 else "sorteado_do_treino",
        "meta": {
            "nome": Path(r["imagem"]).name,
            "imagem": r["imagem"],
            "mascara": r["mascara"],
        },
        "imagem_sha256": sha256(r["imagem"]),
        "mascara_sha256": sha256(r["mascara"]),
    })

plano = {
    "pergunta": "O ganho do EFT se mantém com diferentes suportes Kvasir?",
    "semente": SEMENTE,
    "n_suportes": 20,
    "n_queries": 145,
    "resolucao_encoder": 672,
    "canvas_sam": 1024,
    "checkpoint": str(CHECKPOINT),
    "checkpoint_sha256": sha256(CHECKPOINT),
    "avaliador_sha256": sha256(destino),
    "divisao_sha256": sha256(DADOS / "divisao.json"),
    "manifesto_sha256": sha256(manifesto_path),
    "suportes": suportes,
}

if ARQUIVO_PLANO.exists():
    assert ler_json(ARQUIVO_PLANO) == plano, (
        "A seleção ou algum arquivo mudou. Plano anterior preservado."
    )
else:
    avaliador.salvar(ARQUIVO_PLANO, plano)

display(pd.DataFrame([
    {
        "suporte": s["id"],
        "tipo": s["tipo"],
        "imagem": s["meta"]["nome"],
    }
    for s in suportes
]))

print("EXECUÇÃO:", EXECUCAO)
print("Pares por condição:", 20 * 145)
print("Total, inicial + EFT:", 20 * 145 * 2)
print("Suportes registrados. Avaliação ainda não iniciada.")

In [ ]:
import numpy as np
import torch

assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))
print("EXECUÇÃO:", EXECUCAO, flush=True)

referencia_resumo = ler_json(REFERENCIA / "resumo.json")
linhas = []

for s in plano["suportes"]:
    assert sha256(s["meta"]["imagem"]) == s["imagem_sha256"]
    assert sha256(s["meta"]["mascara"]) == s["mascara_sha256"]

    pasta = EXECUCAO / s["id"]
    retomar = str(pasta) if (pasta / "config.json").exists() else None

    assert not pasta.exists() or retomar is not None, (
        f"Pasta sem config.json: {pasta}. Inspecione antes de continuar."
    )

    print(
        f"\n===== {s['id']} | {s['meta']['nome']} =====",
        flush=True,
    )

    resultado = avaliador.executar(
        piloto=str(PILOTO),
        suporte_override=s["meta"],
        checkpoint_externo=str(CHECKPOINT),
        referencia=str(REFERENCIA),
        destino_novo=str(pasta),
        retomar=retomar,
    )
    resumo = ler_json(Path(resultado) / "resumo.json")
    assert resumo["n"] == 145

    if s["id"] == "S00":
        diferencas = {}
        for etapa in ("ALP", "SAM"):
            for condicao in ("antes", "depois"):
                atual = resumo[etapa]["Dice"][condicao]
                anterior = referencia_resumo[etapa]["Dice"][condicao]
                diferencas[f"{etapa}_{condicao}"] = atual - anterior

        print("Diferenças para a referência:", diferencas, flush=True)
        avaliador.salvar(
            EXECUCAO / "conferencia_suporte_original.json",
            diferencas,
        )
        # Aceita especificamente os resultados do S00 que acabamos
        # de inspecionar. Não libera divergências futuras arbitrárias.
        esperado_4090 = {
            "ALP": {
                "antes": 0.3993660749,
                "depois": 0.5385130833,
            },
            "SAM": {
                "antes": 0.4790349410,
                "depois": 0.6251589416,
            },
        }

        config_s00 = ler_json(Path(resultado) / "config.json")
        assert config_s00["gpu_driver"] == (
            "NVIDIA GeForce RTX 4090, 595.91.07"
        ), "Ambiente diferente daquele que acabamos de conferir."

        for etapa, condicoes in esperado_4090.items():
            for condicao, esperado in condicoes.items():
                observado = resumo[etapa]["Dice"][condicao]
                assert abs(observado - esperado) <= 1e-8, (
                    f"S00 mudou: {etapa}/{condicao}: {observado}"
                )

        avaliador.salvar(
            EXECUCAO / "conferencia_suporte_original_revisada.json",
            {
                "decisao": "Prosseguir com comparação pareada na RTX 4090",
                "referencia": str(REFERENCIA),
                "diferencas_dice": diferencas,
                "valores_s00_4090": esperado_4090,
                "observacao": (
                    "Diferença observada principalmente no SAM inicial. "
                    "Causa não comprovada. Resultados da referência "
                    "preservados; comparação multissuporte usa a execução atual."
                ),
            },
        )
        print("S00 conferido. Prosseguindo com os demais suportes.", flush=True)

    linha = {
        "suporte": s["id"],
        "tipo": s["tipo"],
        "imagem": s["meta"]["nome"],
    }
    for etapa in ("ALP", "SAM"):
        linha[f"{etapa}_inicial"] = resumo[etapa]["Dice"]["antes"]
        linha[f"{etapa}_EFT"] = resumo[etapa]["Dice"]["depois"]
        linha[f"{etapa}_ganho"] = (
            linha[f"{etapa}_EFT"] - linha[f"{etapa}_inicial"]
        )
    linhas.append(linha)

    tabela = pd.DataFrame(linhas)
    temporario = EXECUCAO / "por_suporte.csv.tmp"
    tabela.to_csv(temporario, index=False)
    os.replace(temporario, EXECUCAO / "por_suporte.csv")

display(tabela.round(4))

# O suporte original é fixo e reservado; os demais foram sorteados.
# Mostramos os dois agrupamentos separadamente.
for nome, grupo in (
    ("Todos os 20 suportes", tabela),
    ("Somente os 19 suportes sorteados", tabela[tabela["suporte"] != "S00"]),
):
    print(f"\n{nome}")
    for etapa in ("ALP", "SAM"):
        inicial = grupo[f"{etapa}_inicial"]
        eft = grupo[f"{etapa}_EFT"]
        ganho = grupo[f"{etapa}_ganho"]
        print(
            f"{etapa}: inicial={inicial.mean():.4f} ± {inicial.std(ddof=1):.4f}"
            f" | EFT={eft.mean():.4f} ± {eft.std(ddof=1):.4f}"
            f" | ganho médio={100 * ganho.mean():+.2f} pp"
            f" | EFT melhor em {int((ganho > 0).sum())}/{len(grupo)} suportes"
        )

print("\nConcluído:", EXECUCAO)

In [ ]:
from pathlib import Path
import json

BASE = Path("/workspace/ProtoSAM-repro/reproducao")
ATUAL = BASE / "multissuporte_tiny_eft2000_kvasir_v1/S00"
REF = BASE / "eft_tiny_referencia_driver_atual_20261012T215801165279Z"

def ler(p):
    return json.loads(p.read_text(encoding="utf-8"))

atual = ler(ATUAL / "resumo.json")
referencia = ler(REF / "resumo.json")

print("=== DICE: REFERÊNCIA × ATUAL ===")
for etapa in ("ALP", "SAM"):
    for condicao in ("antes", "depois"):
        a = atual[etapa]["Dice"][condicao]
        r = referencia[etapa]["Dice"][condicao]
        print(
            f"{etapa:3s} | {condicao:6s} | "
            f"referência={r:.10f} | atual={a:.10f} | "
            f"diferença={a-r:+.10f}"
        )

cfg_atual = ler(ATUAL / "config.json")
cfg_ref = ler(REF / "config.json")

print("\n=== DIFERENÇAS DE CONFIGURAÇÃO ===")
for chave in sorted(set(cfg_atual) | set(cfg_ref)):
    if chave == "fontes_sha256":
        continue
    a = cfg_atual.get(chave, "<ausente>")
    r = cfg_ref.get(chave, "<ausente>")
    if a != r:
        print(f"\n{chave}")
        if chave == "episodios":
            print("As listas de queries diferem.")
        else:
            print("  Referência:", r)
            print("  Atual:     ", a)

print("\n=== FONTES PRESENTES NAS DUAS CONFIGURAÇÕES ===")
fontes_a = cfg_atual.get("fontes_sha256", {})
fontes_r = cfg_ref.get("fontes_sha256", {})
mudaram = [
    p for p in fontes_a.keys() & fontes_r.keys()
    if fontes_a[p] != fontes_r[p]
]
print("Fontes compartilhadas com hash diferente:", mudaram)

print("\n=== REGISTROS SALVOS ===")
for nome, pasta in (("Referência", REF), ("Atual", ATUAL)):
    for condicao in ("antes", "depois"):
        quantidade = len(list((pasta / condicao / "registros").glob("*.json")))
        print(f"{nome} / {condicao}: {quantidade}")